# 第 3 讲｜Shaw 相对位置编码：从 QKᵀ 的位置表开始

本讲讨论**同一个序列的 Self-Attention（自注意力）**，先不加因果掩码。Q、K、V 都由同一个输入序列生成；**不讨论 Cross-Attention**。最后的类会附带可选的因果掩码。

我们只解决一条递进关系：**已有的绝对位置编码怎样进入 Attention → 为什么为每对 Query–Key 再建一张相对距离表 → Shaw 如何使用这张表。**

## 一、先接上上一讲：绝对位置已经能影响 Attention

在原始 Transformer 中，先把内容向量 `x_i` 和绝对位置向量 `p_i` 相加：

$$h_i=x_i+p_i$$

同一个序列的 `h` 分别经过线性投影得到 Q、K、V：

$$q_i=W_Qh_i,\qquad k_j=W_Kh_j,\qquad v_j=W_Vh_j$$

标准缩放点积注意力的分数是：

$$s_{ij}=\frac{q_i^\top k_j}{\sqrt{d_k}}$$

其中 `i` 是 **Query 的行号**，`j` 是 **Key 的列号**。`s_ij` 是分数，尚不是 Softmax 后的权重。

**为什么已经有位置？** 为了只看清这一点，暂时假设 Q/K 的投影是单位矩阵（实际训练时通常不是），则：

$$
\begin{aligned}
q_i^\top k_j
&=(x_i+p_i)^\top(x_j+p_j)\\
&=\underbrace{x_i^\top x_j}_{\text{内容与内容}}
+\underbrace{x_i^\top p_j}_{\text{内容与位置}}
+\underbrace{p_i^\top x_j}_{\text{位置与内容}}
+\underbrace{p_i^\top p_j}_{\text{位置与位置}}.
\end{aligned}
$$

特别是最后一项。对于单个频率 `ω` 的 Sinusoidal 位置向量：

$$p_t=[\sin(\omega t),\cos(\omega t)]$$

利用 `cos(A−B)=cos(A)cos(B)+sin(A)sin(B)`，有：

$$\boxed{p_i^\top p_j=\cos(\omega(i-j))}$$

也就是说，**原始 Transformer 加入 Sinusoidal PE 后，确实能够利用相对距离信息**。但实际经过可训练 Q/K 投影后，完整分数不一定只依赖 `i-j`。

**那 Shaw 为什么还要另做相对位置？** 因为它选择更直接的办法：既然注意力分数是在比较一对 Token，就先求出这一对 Token 的**相对距离**，再为这段距离查一个可学习向量，直接参与注意力计算。

In [1]:
import math                          # 使用 sqrt() 计算 Attention 缩放系数
import torch                         # PyTorch：创建张量、矩阵乘法
import torch.nn as nn               # PyTorch 模块：Linear、Embedding

# 这里验证上一讲的一个事实：Sinusoidal PE 的位置向量内积与 i-j 有关。
omega = torch.tensor(0.5, dtype=torch.float64)  # 固定一个频率 ω=0.5

def pos_pair(pos):
    """输入位置编号 pos，输出二维向量 [sin(ω·pos), cos(ω·pos)]。"""
    pos = torch.as_tensor(pos, dtype=torch.float64) # 转为浮点张量，便于 sin/cos 计算
    sin_val = torch.sin(omega * pos)                # 计算 sin(ω·pos)
    cos_val = torch.cos(omega * pos)                # 计算 cos(ω·pos)
    return torch.stack([sin_val, cos_val], dim=-1) # 在最后一维拼成两个数：shape=(2,)

# (1,2) 与 (10,11) 的绝对位置不同，但两对位置相差的距离都是 1。
score_1 = torch.dot(pos_pair(1), pos_pair(2))      # (2,) 点乘 (2,) -> 标量
score_2 = torch.dot(pos_pair(10), pos_pair(11))    # 同样计算另一对位置的内积
print('位置 (1,2) 的内积：', score_1.item())
print('位置 (10,11) 的内积：', score_2.item())
assert torch.allclose(score_1, score_2)           # 距离相同 -> 这个位置内积相同

# 注意：这只证明“这一项”体现了相对距离。
# 不代表 Q、K 经任意训练投影后，整个 Attention Score 一定只由距离决定。

位置 (1,2) 的内积： 0.8775825618903728
位置 (10,11) 的内积： 0.8775825618903726


## 二、先看懂 QKᵀ 的每一行、每一列（对应你手绘的矩阵）

现在只有三个 Token，分别位于 `P1、P2、P3`。为方便手写矩阵，每个 Query/Key 都只有两个特征。

**下面只为了展示矩阵乘法，假设 Q=K**；真实 Self-Attention 的 Q/K 往往通过不同参数投影得到，并不要求相等。

$$Q=\begin{array}{c|cc}&\text{特征1}&\text{特征2}\\\hline P_1&a&b\\P_2&c&d\\P_3&e&f\end{array}$$

$$K^\top=\begin{array}{c|ccc}&P_1&P_2&P_3\\\hline \text{特征1}&a&c&e\\\text{特征2}&b&d&f\end{array}$$

- `Q` 是 `(3,2)`：**每一行**是一个位置的 Query。
- `Kᵀ` 是 `(2,3)`：**每一列**是一个位置的 Key。
- 因此 `Q @ K.T` 得到 `(3,3)`：第 `(i,j)` 格是在比较**第 i 个 Query 与第 j 个 Key**。

$$QK^\top=\begin{bmatrix}
a^2+b^2&ac+bd&ae+bf\\
ac+bd&c^2+d^2&ce+df\\
ae+bf&ce+df&e^2+f^2
\end{bmatrix}$$

例如第二行第三列 `ce+df`，就是 `P2` 的 Query 看 `P3` 的 Key 的内容匹配值。真实 Attention Score 还会除以 `√d_k`。

### 既然 Attention 每格都对应一对位置，就给每格计算一个距离

位置编号：`P1→0`、`P2→1`、`P3→2`。规定**Key 位置减 Query 位置**，即 `D[i,j]=j-i`。

先按**行**重复 Query 的位置（同一行属于同一个 Query）：

$$I=\begin{bmatrix}0&0&0\\1&1&1\\2&2&2\end{bmatrix}$$

再按**列**重复 Key 的位置（同一列属于同一个 Key）：

$$J=\begin{bmatrix}0&1&2\\0&1&2\\0&1&2\end{bmatrix}$$

于是逐格相减：

$$\boxed{D=J-I=\begin{bmatrix}0&1&2\\-1&0&1\\-2&-1&0\end{bmatrix}}$$

**和 Attention 的格子一一对齐**：`QKᵀ` 的第二行第三列是 `P2` 对 `P3` 的分数；`D` 同一个格子是 `2−1=+1`，表示 Key 在 Query **右侧 1 个位置**。

Q的第一行是[0,0,0]，减去K的第一行[0,1,2]，你会发现刚好就是第一个Q，对于Key中每个维度的相对位置。这就是为什么I，J矩阵要这么设计的原因。

注意：**`D` 不是 Attention Score，也不能直接拿它当成 Shaw 的位置向量。** 它目前只是一张“每对 Token 相隔多远”的整数表。下一节才会用距离查可学习向量。

In [2]:
# 当前实验：长度为 3 的【双向 Self-Attention】，不加因果掩码。
# 目的：把刚才的 QKᵀ 和相对位置矩阵 D，用实际数字手工对齐。

X = torch.tensor([[1., 2.],                  # P1：位置编号 0，内容向量 [1,2]
                  [3., 4.],                  # P2：位置编号 1，内容向量 [3,4]
                  [5., 6.]])                 # P3：位置编号 2，内容向量 [5,6]
Q = X                                        # 仅为示范矩阵乘法，假设 Q = X
K = X                                        # 仅为示范，假设 K = X；实际一般通过 W_K 投影
print('Q（每行是一个 Query）：\n', Q)          # Q.shape=(3,2)
print('K.T（每列是一个 Key）：\n', K.T)       # K.T.shape=(2,3)

content_score = Q @ K.T                     # (3,2)@(2,3)->(3,3)；逐对计算 Query·Key
print('QKᵀ（暂未除 √d_k）：\n', content_score)
print('P2 看 P3 的内容分数：', content_score[1, 2].item()) # (3×5)+(4×6)=39

position = torch.arange(3)                  # [0,1,2]：三个 Token 的位置；shape=(3,)
L = position.numel()                        # 序列长度 L=3
query_pos = position[:, None].repeat(1, L)  # (3,1) 每行复制 3 次 -> (3,3)，行 i 都是 i。 
key_pos = position[None, :].repeat(L, 1)    # (1,3) 每列复制 3 次 -> (3,3)，列 j 都是 j。 
print('Query 的位置矩阵 I：\n', query_pos)
print('Key 的位置矩阵 J：\n', key_pos)

relative_distance = key_pos - query_pos     # 对应手算 J-I；每格是 Key 位置减 Query 位置
print('相对距离 D=J-I：\n', relative_distance)
print('P2 看 P3 的距离：', relative_distance[1, 2].item()) # j=2,i=1，得到 +1

# 手算理解后，就不需要真的用 repeat 把位置复制很多遍了：改用广播。
key_row = position[None, :]                # 在前面增加维度：(3,)->(1,3)，一行 Key 位置。这个地方就是利用广播的复制性质造出上面说的I
query_column = position[:, None]           # 在最后增加维度：(3,)->(3,1)，一列 Query 位置。这个地方就是利用广播的复制性质造出上面说的J
D_broadcast = key_row - query_column       # (1,3)-(3,1)，广播后得到 (3,3)，每格 j-i。相对位置矩阵。
assert torch.equal(relative_distance, D_broadcast)  # 验证广播与手算完全相同
assert content_score[1, 2].item() == 39.0
assert D_broadcast[1, 2].item() == 1

Q（每行是一个 Query）：
 tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])
K.T（每列是一个 Key）：
 tensor([[1., 3., 5.],
        [2., 4., 6.]])
QKᵀ（暂未除 √d_k）：
 tensor([[ 5., 11., 17.],
        [11., 25., 39.],
        [17., 39., 61.]])
P2 看 P3 的内容分数： 39.0
Query 的位置矩阵 I：
 tensor([[0, 0, 0],
        [1, 1, 1],
        [2, 2, 2]])
Key 的位置矩阵 J：
 tensor([[0, 1, 2],
        [0, 1, 2],
        [0, 1, 2]])
相对距离 D=J-I：
 tensor([[ 0,  1,  2],
        [-1,  0,  1],
        [-2, -1,  0]])
P2 看 P3 的距离： 1



## 三、Shaw 如何将相对位置引入 Attention？

本节讨论**双向 Self-Attention**，Q、K、V 均来自同一个输入序列。

### 1. 从 Learned Absolute PE 到 Shaw RPE

Learned Absolute PE 根据 Token 的绝对位置查表，并在生成 Q、K、V 之前加到输入上：

$$
p_i=E_{\mathrm{abs}}[i],\qquad h_i=x_i+p_i
$$

Shaw 沿用了**可学习位置向量表**的思想，但将查表依据从绝对位置 $i$ 改成了相对距离 $j-i$。

为什么不能像 Learned Absolute PE 一样，提前给每个 Token 加一个相对位置向量？

例如，三个 Token 的位置分别为 0、1、2：

| Query | Key | 相对距离 |
|---|---|---|
| P1（0） | P2（1） | +1 |
| P3（2） | P2（1） | -1 |

两次关注的都是 P2，但它相对于不同 Query 的距离不同。

因此，**相对位置属于 Query–Key 之间的关系，不能提前为每个 Token 固定一个相对位置向量。**

Shaw 需要在计算每一对 Query–Key 时，根据它们的相对距离查表。

### 2. 如何根据相对距离查表？

前面已经得到相对距离矩阵：

$$
D_{ij}=j-i
$$

例如 P2 作为 Query 时，对应的距离为：

$$
D[1,:]=[-1,0,+1]
$$

距离本身只表示方向和远近。如果直接将整数距离加到 Attention Score，就会人为规定正距离比负距离提供更大的分数。

因此，Shaw 为每种相对距离学习一个向量：

| 距离 | 查到的向量 |
|---|---|
| -1 | $a^K_{-1}$ |
| 0 | $a^K_0$ |
| +1 | $a^K_{+1}$ |

这些向量是模型的可训练参数，**相同的距离在同一张表中查到相同的向量**。

为控制参数量，Shaw 设置最大相对距离 $r$：

$$
a^K_{ij}=E_K[\operatorname{clip}(j-i,-r,r)+r]
$$

其中：

- `clamp(-r,r)`：将超出范围的距离裁剪到 $[-r,r]$。
- `+r`：将距离转换为非负索引，供 `nn.Embedding` 查表。
- $E_K$：大小为 $(2r+1,d_k)$ 的可学习位置向量表。

### 3. 相对位置向量如何参与 Attention？

普通 Self-Attention 的分数为：

$$
s_{ij}=\frac{q_i^\top k_j}{\sqrt{d_k}}
$$

Shaw 根据相对距离查出 $a^K_{ij}$，**先将它加到当前 Query–Key 匹配使用的 Key 向量上，再与 Query 做点积**：

$$
\boxed{
s_{ij}^{\mathrm{Shaw}}
=\frac{q_i^\top(k_j+a^K_{ij})}{\sqrt{d_k}}
}
$$

展开得到：

$$
s_{ij}^{\mathrm{Shaw}}
=
\underbrace{\frac{q_i^\top k_j}{\sqrt{d_k}}}_{\text{内容分数}}
+
\underbrace{\frac{q_i^\top a^K_{ij}}{\sqrt{d_k}}}_{\text{位置分数}}
$$

为什么使用向量而不是直接学习一个标量？

因为同一个相对距离对应同一个位置向量，但不同 Query 与它的点积结果可能不同：

$$
q_0^\top a^K_{+1}\neq q_1^\top a^K_{+1}
$$

所以，**相对距离决定查哪个向量，Query 与该向量的点积决定位置分数。**

得到新的 Attention Score 后，仍然按照普通 Self-Attention 的方式计算权重：

$$
\alpha_{ij}
=\operatorname{softmax}_j(s_{ij}^{\mathrm{Shaw}})
$$

例如 P2 作为 Query 时，距离 $[-1,0,+1]$ 对应三个位置向量，分别参与三个 Attention Score 的计算，再经过 Softmax 得到权重。

### 4. 为什么 Value 也需要相对位置向量？

Learned Absolute PE 在生成 Q、K、V 之前就加入位置向量，因此 Value 本身也包含绝对位置信息：

$$
v_j=(x_j+p_j)W_V
$$

但对于同一个 Key，不同 Query 使用的仍是同一个 $v_j$，无法直接体现它们之间不同的相对距离。

Shaw 因此另外使用一张可学习表 $E_V$，将相对位置向量加入 Value 聚合：

$$
\boxed{
z_i=\sum_j\alpha_{ij}(v_j+a^V_{ij})
}
$$

其中，$\alpha_{ij}$ 使用前面 Key 分支计算的权重。

Shaw 使用两张独立的位置表，是因为它们参与的计算不同：

- **$E_K$**：位置向量参与 Query–Key 点积，影响 Attention Score 和权重。
- **$E_V$**：位置向量参与 Value 加权求和，影响最终输出。

两张表可以分别学习适合各自计算的位置表示，而不必共享参数。

**总结：Shaw RPE 沿用可学习向量表的思想，但根据相对距离查表，并在 Attention 的评分和 Value 聚合阶段分别引入相对位置向量。**
